# Dubai Real Estate Market Analysis — 2020 to 2024
**Transaction-level analysis of DLD property sales across 12 districts**  
*12,892 transactions · Price per sqft · Buyer nationality · Payment methods · Luxury segment*


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DARK    = "#0d1117"
ACC1    = "#00d4aa"
ACC2    = "#f7b731"
ACC3    = "#e74c3c"
ACC4    = "#3498db"
PALETTE = [ACC1, ACC2, ACC3, ACC4, "#9b59b6", "#e67e22",
           "#1abc9c", "#e91e63", "#2196f3", "#ff5722", "#8bc34a", "#ff9800"]


## 1. Data Source
Simulates a pull from the Dubai Land Department open data portal. The dataset mirrors real DLD transaction structure: OHLCV-equivalent fields for property — district, type, size, price per sqft, payment method, buyer nationality, date.

In [2]:
districts = {
    "Dubai Marina":            {"base_psf": 1450, "vol_weight": 0.12, "type_mix": {"Apartment": 0.85, "Villa": 0.05, "Penthouse": 0.10}},
    "Downtown Dubai":          {"base_psf": 1800, "vol_weight": 0.10, "type_mix": {"Apartment": 0.70, "Penthouse": 0.20, "Villa": 0.10}},
    "Palm Jumeirah":           {"base_psf": 2400, "vol_weight": 0.07, "type_mix": {"Villa": 0.45, "Apartment": 0.35, "Penthouse": 0.20}},
    "Business Bay":            {"base_psf": 1350, "vol_weight": 0.11, "type_mix": {"Apartment": 0.90, "Penthouse": 0.10}},
    "Jumeirah Village Circle": {"base_psf": 950,  "vol_weight": 0.13, "type_mix": {"Apartment": 0.80, "Villa": 0.20}},
    "Arabian Ranches":         {"base_psf": 1100, "vol_weight": 0.06, "type_mix": {"Villa": 0.95, "Apartment": 0.05}},
    "DIFC":                    {"base_psf": 2100, "vol_weight": 0.05, "type_mix": {"Apartment": 0.60, "Penthouse": 0.40}},
    "Al Barsha":               {"base_psf": 850,  "vol_weight": 0.08, "type_mix": {"Apartment": 0.75, "Villa": 0.25}},
    "Dubai Hills Estate":      {"base_psf": 1600, "vol_weight": 0.09, "type_mix": {"Villa": 0.55, "Apartment": 0.45}},
    "Jumeirah Lake Towers":    {"base_psf": 1050, "vol_weight": 0.09, "type_mix": {"Apartment": 0.95, "Penthouse": 0.05}},
    "Meydan":                  {"base_psf": 1250, "vol_weight": 0.05, "type_mix": {"Villa": 0.60, "Apartment": 0.40}},
    "Silicon Oasis":           {"base_psf": 720,  "vol_weight": 0.05, "type_mix": {"Apartment": 0.85, "Villa": 0.15}},
}

nationalities = ["UAE National", "Indian", "British", "Russian", "Chinese",
                 "Pakistani", "American", "Saudi", "German", "Egyptian"]
nat_weights   = [0.12, 0.18, 0.10, 0.12, 0.08, 0.09, 0.06, 0.08, 0.05, 0.12]

macro_mult = {
    "2020Q1": 0.97, "2020Q2": 0.88, "2020Q3": 0.90, "2020Q4": 0.93,
    "2021Q1": 0.95, "2021Q2": 1.00, "2021Q3": 1.06, "2021Q4": 1.12,
    "2022Q1": 1.18, "2022Q2": 1.25, "2022Q3": 1.30, "2022Q4": 1.34,
    "2023Q1": 1.36, "2023Q2": 1.38, "2023Q3": 1.41, "2023Q4": 1.43,
    "2024Q1": 1.46, "2024Q2": 1.48, "2024Q3": 1.50, "2024Q4": 1.52,
}

np.random.seed(7)
quarters = pd.period_range("2020-Q1", "2024-Q4", freq="Q")
rows = []

for q in quarters:
    qkey = f"{q.year}Q{q.quarter}"
    mult = macro_mult.get(qkey, 1.0)
    month_start = (q.quarter - 1) * 3 + 1

    for dist, cfg in districts.items():
        n_txn = max(int(np.random.normal(cfg["vol_weight"] * 600, 30)), 20)

        for _ in range(n_txn):
            prop_type  = np.random.choice(list(cfg["type_mix"].keys()),
                                           p=list(cfg["type_mix"].values()))
            size_range = {"Apartment": (450, 2200), "Villa": (2000, 8000), "Penthouse": (1800, 5000)}
            lo, hi     = size_range[prop_type]
            area_sqft  = round(np.random.uniform(lo, hi), 0)
            psf        = round(cfg["base_psf"] * mult * np.random.normal(1.0, 0.08), 0)
            price_aed  = round(area_sqft * psf / 1000) * 1000

            cash_p     = 0.48 if q.year <= 2021 else 0.38
            payment    = np.random.choice(["Cash", "Mortgage"], p=[cash_p, 1 - cash_p])
            buyer_nat  = np.random.choice(nationalities, p=nat_weights)
            month      = month_start + np.random.randint(0, 3)
            txn_date   = pd.Timestamp(year=q.year, month=month, day=np.random.randint(1, 28))

            rows.append({
                "transaction_date":      txn_date,
                "year":                  q.year,
                "quarter":               f"Q{q.quarter}",
                "district":              dist,
                "property_type":         prop_type,
                "area_sqft":             area_sqft,
                "price_per_sqft":        psf,
                "transaction_value_aed": price_aed,
                "payment_method":        payment,
                "buyer_nationality":     buyer_nat,
            })

raw = pd.DataFrame(rows).sort_values("transaction_date").reset_index(drop=True)

null_idx = np.random.choice(raw.index, size=int(len(raw) * 0.02), replace=False)
raw.loc[null_idx[:len(null_idx)//2], "buyer_nationality"] = np.nan
raw.loc[null_idx[len(null_idx)//2:], "area_sqft"]         = np.nan

print(f"Raw dataset: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Date range : {raw['transaction_date'].min().date()} → {raw['transaction_date'].max().date()}")
raw.head(6)


Raw dataset: 12,892 rows × 10 columns
Date range : 2020-01-01 → 2024-12-27


,transaction_date,year,quarter,district,property_type,area_sqft,price_per_sqft,transaction_value_aed,payment_method,buyer_nationality
0,2020-01-01,2020,Q1,Palm Jumeirah,Villa,"4,153.00","2,475.00",10279000,Cash,Chinese
1,2020-01-01,2020,Q1,Jumeirah Village Circle,Villa,"2,493.00",935.00,2331000,Mortgage,German
2,2020-01-01,2020,Q1,Jumeirah Lake Towers,Penthouse,"4,655.00",783.00,3645000,Mortgage,American
3,2020-01-01,2020,Q1,Business Bay,Apartment,"1,708.00","1,381.00",2359000,Mortgage,Indian
4,2020-01-01,2020,Q1,Business Bay,Apartment,"1,015.00","1,197.00",1215000,Mortgage,British
5,2020-01-01,2020,Q1,Palm Jumeirah,Villa,"6,063.00","2,441.00",14800000,Mortgage,Egyptian


## 2. Data Cleaning

In [3]:
df = raw.copy()

print("Nulls before cleaning:")
print(df.isnull().sum())

df["area_sqft"] = df["area_sqft"].fillna(
    df.groupby(["district", "property_type"])["area_sqft"].transform("median")
)
df["buyer_nationality"] = df["buyer_nationality"].fillna("Unknown")

before = len(df)
df = df[df["transaction_value_aed"] > 50_000]
df = df[df["price_per_sqft"].between(300, 6_000)]
df = df[df["area_sqft"].between(200, 15_000)]
print(f"\nRows removed as outliers: {before - len(df)}")
print(f"Clean dataset: {len(df):,} rows")

print("\nNulls after cleaning:")
print(df.isnull().sum())


Nulls before cleaning:
transaction_date           0
year                       0
quarter                    0
district                   0
property_type              0
area_sqft                129
price_per_sqft             0
transaction_value_aed      0
payment_method             0
buyer_nationality        128
dtype: int64

Rows removed as outliers: 0
Clean dataset: 12,892 rows

Nulls after cleaning:
transaction_date         0
year                     0
quarter                  0
district                 0
property_type            0
area_sqft                0
price_per_sqft           0
transaction_value_aed    0
payment_method           0
buyer_nationality        0
dtype: int64


## 3. Feature Engineering

In [4]:
df["year_quarter"] = df["year"].astype(str) + " " + df["quarter"]
df["value_m_aed"]  = df["transaction_value_aed"] / 1_000_000
df["is_foreign"]   = (df["buyer_nationality"] != "UAE National").astype(int)
df["is_luxury"]    = (df["transaction_value_aed"] >= 5_000_000).astype(int)
df["price_tier"]   = pd.cut(
    df["price_per_sqft"],
    bins=[0, 800, 1200, 1800, 2500, 9999],
    labels=["Budget", "Mid", "Premium", "Luxury", "Ultra"]
)

summary = (
    df.groupby("district")
      .agg(
          transactions     = ("transaction_value_aed", "count"),
          avg_psf          = ("price_per_sqft", "mean"),
          median_ticket_m  = ("value_m_aed", "median"),
          total_volume_b   = ("value_m_aed", "sum"),
          foreign_pct      = ("is_foreign", lambda x: round(x.mean() * 100, 1)),
          luxury_pct       = ("is_luxury",  lambda x: round(x.mean() * 100, 1)),
      )
      .sort_values("avg_psf", ascending=False)
      .round(2)
)
summary


,transactions,avg_psf,median_ticket_m,total_volume_b,foreign_pct,luxury_pct
district,,,,,,
Palm Jumeirah,930,"2,903.25",8.60,"9,245.82",88.80,72.50
DIFC,806,"2,668.42",4.98,"4,602.52",87.30,49.80
Downtown Dubai,1092,"2,220.23",3.81,"5,294.71",87.70,34.30
Dubai Hills Estate,1124,"1,968.93",5.16,"7,491.52",89.70,50.90
Dubai Marina,1351,"1,826.88",2.57,"4,351.25",87.00,12.00
Business Bay,1501,"1,624.59",2.20,"3,797.22",87.50,6.60
Meydan,806,"1,503.11",4.28,"4,281.00",88.00,45.40
Arabian Ranches,887,"1,338.47",6.14,"5,730.68",87.30,64.50
Jumeirah Lake Towers,1138,"1,292.46",1.71,"2,136.14",88.80,2.40


## 4. Analysis & Visualisations

### 4.1 Quarterly Transaction Volume & Price Trend

In [5]:
yq = (
    df.groupby(["year", "quarter"])
      .agg(txn_count=("transaction_value_aed", "count"),
           avg_psf=("price_per_sqft", "mean"),
           total_vol_b=("value_m_aed", "sum"))
      .reset_index()
)
yq["label"] = yq["year"].astype(str) + " " + yq["quarter"]

fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_trace(go.Bar(x=yq["label"], y=yq["txn_count"],
    name="Transaction Count", marker_color=ACC1, opacity=0.85), secondary_y=False)
fig1.add_trace(go.Scatter(x=yq["label"], y=yq["avg_psf"],
    name="Avg Price/sqft (AED)", line=dict(color=ACC2, width=2.5),
    mode="lines+markers", marker=dict(size=6)), secondary_y=True)
fig1.update_layout(
    title="Dubai Real Estate — Quarterly Transaction Volume & Price Trend (2020–2024)",
    template="plotly_dark", paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=460, legend=dict(orientation="h", y=1.08)
)
fig1.update_yaxes(title_text="Transactions", secondary_y=False)
fig1.update_yaxes(title_text="Avg AED / sqft", secondary_y=True)
fig1.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

### 4.2 Price per sqft Heatmap by District & Year

In [ ]:
hm = df.pivot_table(index="district", columns="year",
                    values="price_per_sqft", aggfunc="mean").round(0)

fig2 = go.Figure(go.Heatmap(
    z=hm.values, x=hm.columns.tolist(), y=hm.index.tolist(),
    colorscale="YlOrRd", colorbar=dict(title="AED/sqft"),
    text=np.round(hm.values, 0), texttemplate="%{text:,.0f}",
    hovertemplate="District: %{y}<br>Year: %{x}<br>AED/sqft: %{z:,.0f}<extra></extra>"
))
fig2.update_layout(
    title="Average Price per sqft by District & Year (AED)",
    template="plotly_dark", paper_bgcolor=DARK,
    plot_bgcolor=DARK, height=480
)
fig2.show()


### 4.3 Buyer Nationality — Treemap

In [ ]:
nat = (
    df[df["buyer_nationality"] != "Unknown"]
      .groupby("buyer_nationality")
      .agg(count=("transaction_value_aed", "count"),
           total_value=("value_m_aed", "sum"))
      .reset_index()
)
nat["avg_ticket"] = nat["total_value"] / nat["count"]

fig3 = px.treemap(nat, path=["buyer_nationality"], values="count",
    color="avg_ticket", color_continuous_scale="teal",
    title="Buyer Nationality — Transaction Count & Avg Ticket Size (AED M)",
    hover_data={"total_value": ":.1f", "avg_ticket": ":.2f"})
fig3.update_layout(
    template="plotly_dark", paper_bgcolor=DARK, height=480,
    coloraxis_colorbar=dict(title="Avg Ticket (AED M)")
)
fig3.show()


### 4.4 Property Type Mix by Year

In [ ]:
pt = df.groupby(["year", "property_type"]).size().reset_index(name="count")
pt["pct"] = pt["count"] / pt.groupby("year")["count"].transform("sum") * 100

fig4 = px.bar(pt, x="year", y="pct", color="property_type",
    barmode="stack",
    title="Property Type Mix by Year (%)",
    color_discrete_sequence=PALETTE,
    labels={"pct": "Share (%)", "property_type": "Type"},
    template="plotly_dark")
fig4.update_layout(paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=420, legend=dict(orientation="h", y=1.08))
fig4.show()


### 4.5 Cash vs Mortgage Split

In [ ]:
pay = df.groupby(["year", "quarter", "payment_method"]).size().reset_index(name="count")
pay["yq"] = pay["year"].astype(str) + " " + pay["quarter"]
pay_piv = pay.pivot_table(index="yq", columns="payment_method",
                          values="count", aggfunc="sum").fillna(0)
pay_piv["cash_pct"] = pay_piv["Cash"] / (pay_piv["Cash"] + pay_piv["Mortgage"]) * 100

fig5 = make_subplots(specs=[[{"secondary_y": True}]])
fig5.add_trace(go.Bar(x=pay_piv.index, y=pay_piv["Cash"],
    name="Cash", marker_color=ACC1, opacity=0.8), secondary_y=False)
fig5.add_trace(go.Bar(x=pay_piv.index, y=pay_piv["Mortgage"],
    name="Mortgage", marker_color=ACC4, opacity=0.8), secondary_y=False)
fig5.add_trace(go.Scatter(x=pay_piv.index, y=pay_piv["cash_pct"],
    name="Cash %", line=dict(color=ACC2, width=2.5, dash="dot"),
    mode="lines+markers"), secondary_y=True)
fig5.update_layout(
    barmode="stack",
    title="Cash vs Mortgage Transaction Split (Quarterly)",
    template="plotly_dark", paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=440, legend=dict(orientation="h", y=1.08)
)
fig5.update_yaxes(title_text="Transaction Count", secondary_y=False)
fig5.update_yaxes(title_text="Cash %", secondary_y=True)
fig5.show()


### 4.6 Luxury Segment Growth

In [ ]:
lux = (
    df.groupby(["year", "quarter"])
      .agg(luxury_count=("is_luxury", "sum"), total=("is_luxury", "count"))
      .reset_index()
)
lux["luxury_pct"] = lux["luxury_count"] / lux["total"] * 100
lux["yq"] = lux["year"].astype(str) + " " + lux["quarter"]

fig6 = make_subplots(specs=[[{"secondary_y": True}]])
fig6.add_trace(go.Bar(x=lux["yq"], y=lux["luxury_count"],
    name="Luxury Transactions (≥ 5M AED)", marker_color=ACC3, opacity=0.8), secondary_y=False)
fig6.add_trace(go.Scatter(x=lux["yq"], y=lux["luxury_pct"],
    name="Luxury Share %", line=dict(color=ACC2, width=2.5),
    mode="lines+markers"), secondary_y=True)
fig6.update_layout(
    title="Luxury Segment Growth — Transactions ≥ 5M AED",
    template="plotly_dark", paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=440, legend=dict(orientation="h", y=1.08)
)
fig6.update_yaxes(title_text="Count", secondary_y=False)
fig6.update_yaxes(title_text="Share %", secondary_y=True)
fig6.show()


### 4.7 District Price Appreciation Index (2020 = 100)

In [ ]:
idx = (
    df.groupby(["district", "year"])["price_per_sqft"]
      .mean()
      .unstack()
      .apply(lambda r: r / r[2020] * 100, axis=1)
)

fig7 = go.Figure()
for i, dist in enumerate(idx.index):
    fig7.add_trace(go.Scatter(
        x=idx.columns, y=idx.loc[dist], name=dist,
        line=dict(color=PALETTE[i % len(PALETTE)], width=2),
        mode="lines+markers", marker=dict(size=7)
    ))
fig7.add_hline(y=100, line_dash="dash", line_color="white",
               opacity=0.3, annotation_text="2020 baseline")
fig7.update_layout(
    title="District Price Appreciation Index (2020 = 100)",
    template="plotly_dark", paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=500, xaxis_title="Year", yaxis_title="Index",
    legend=dict(orientation="h", y=-0.22, x=0)
)
fig7.show()


### 4.8 Foreign Investor Concentration vs Price

In [ ]:
foreign = (
    df[df["is_foreign"] == 1]
      .groupby("district")
      .agg(foreign_txns=("is_foreign", "count"),
           avg_psf=("price_per_sqft", "mean"),
           avg_ticket=("value_m_aed", "mean"))
      .reset_index()
)
total_per_dist = df.groupby("district").size().reset_index(name="total")
foreign = foreign.merge(total_per_dist, on="district")
foreign["foreign_pct"] = foreign["foreign_txns"] / foreign["total"] * 100

fig8 = px.scatter(foreign, x="avg_psf", y="foreign_pct",
    size="avg_ticket", color="district", text="district",
    title="Foreign Investor Share vs Avg Price/sqft by District",
    labels={"avg_psf": "Avg AED/sqft", "foreign_pct": "Foreign Buyer %",
            "avg_ticket": "Avg Ticket (AED M)"},
    color_discrete_sequence=PALETTE,
    template="plotly_dark", size_max=45)
fig8.update_traces(textposition="top center", textfont_size=9)
fig8.update_layout(paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=520, showlegend=False)
fig8.show()


### 4.9 Transaction Volume by Price Tier

In [ ]:
tier_data = (
    df.groupby(["year", "price_tier"])
      .agg(volume=("value_m_aed", "sum"))
      .reset_index()
)

fig9 = px.bar(tier_data, x="year", y="volume", color="price_tier",
    barmode="group",
    title="Transaction Volume by Price Tier (AED Million)",
    color_discrete_map={
        "Budget": "#2ecc71", "Mid": "#3498db", "Premium": "#f39c12",
        "Luxury": "#e74c3c", "Ultra": "#9b59b6"
    },
    labels={"volume": "Total Volume (AED M)", "price_tier": "Tier"},
    template="plotly_dark")
fig9.update_layout(paper_bgcolor=DARK, plot_bgcolor=DARK,
    height=440, legend=dict(orientation="h", y=1.08))
fig9.show()


## 5. Business Insights

In [ ]:
total_vol  = df["value_m_aed"].sum() / 1000
avg_psf_20 = df[df["year"]==2020]["price_per_sqft"].mean()
avg_psf_24 = df[df["year"]==2024]["price_per_sqft"].mean()
psf_growth = (avg_psf_24 - avg_psf_20) / avg_psf_20 * 100

top_foreign = (df[df["is_foreign"]==1]
               .groupby("district")["is_foreign"].count()
               .idxmax())
foreign_share_24 = df[df["year"]==2024]["is_foreign"].mean()*100
lux_share_24 = df[df["year"]==2024]["is_luxury"].mean()*100
lux_share_20 = df[df["year"]==2020]["is_luxury"].mean()*100

cash_pct_20 = df[df["year"]==2020]["payment_method"].eq("Cash").mean()*100
cash_pct_24 = df[df["year"]==2024]["payment_method"].eq("Cash").mean()*100

top_nat = (df[df["buyer_nationality"]!="Unknown"]
           .groupby("buyer_nationality")["value_m_aed"].sum()
           .nlargest(3).index.tolist())

print("=" * 60)
print("INSIGHT 1 — Market Recovery & Sustained Price Growth")
print("=" * 60)
print(f"  Price/sqft grew {psf_growth:.1f}% from 2020 to 2024")
print(f"  Total volume analysed: AED {total_vol:.1f}B")
print(f"  2020 avg psf: AED {avg_psf_20:,.0f} → 2024: AED {avg_psf_24:,.0f}")

print()
print("=" * 60)
print("INSIGHT 2 — Foreign Capital is the Dominant Demand Driver")
print("=" * 60)
print(f"  Foreign buyer share in 2024: {foreign_share_24:.1f}%")
print(f"  Highest foreign concentration district: {top_foreign}")
print(f"  Top 3 nationalities by volume: {', '.join(top_nat)}")

print()
print("=" * 60)
print("INSIGHT 3 — Luxury Segment Outpacing Affordable Tiers")
print("=" * 60)
print(f"  Luxury (≥5M AED) share: {lux_share_20:.1f}% in 2020 → {lux_share_24:.1f}% in 2024")

print()
print("=" * 60)
print("INSIGHT 4 — Mortgage Adoption Rising, Reducing Cash Dependency")
print("=" * 60)
print(f"  Cash share: {cash_pct_20:.1f}% in 2020 → {cash_pct_24:.1f}% in 2024")
print(f"  Suggests growing confidence in UAE banking/financing products")

print()
print("=" * 60)
print("INSIGHT 5 — District Divergence Creates Arbitrage Opportunities")
print("=" * 60)
div = (df.groupby("district")["price_per_sqft"].mean()
         .sort_values(ascending=False))
print(f"  Highest PSF: {div.index[0]} at AED {div.iloc[0]:,.0f}/sqft")
print(f"  Lowest PSF:  {div.index[-1]} at AED {div.iloc[-1]:,.0f}/sqft")
print(f"  Price gap ratio: {div.iloc[0]/div.iloc[-1]:.1f}x")


INSIGHT 1 — Market Recovery & Sustained Price Growth
  Price/sqft grew 59.8% from 2020 to 2024
  Total volume analysed: AED 54.2B
  2020 avg psf: AED 1,261 → 2024: AED 2,015

INSIGHT 2 — Foreign Capital is the Dominant Demand Driver
  Foreign buyer share in 2024: 87.7%
  Highest foreign concentration district: Business Bay
  Top 3 nationalities by volume: Indian, Egyptian, UAE National

INSIGHT 3 — Luxury Segment Outpacing Affordable Tiers
  Luxury (≥5M AED) share: 20.9% in 2020 → 32.1% in 2024

INSIGHT 4 — Mortgage Adoption Rising, Reducing Cash Dependency
  Cash share: 47.2% in 2020 → 38.6% in 2024
  Suggests growing confidence in UAE banking/financing products

INSIGHT 5 — District Divergence Creates Arbitrage Opportunities
  Highest PSF: Palm Jumeirah at AED 2,903/sqft
  Lowest PSF:  Silicon Oasis at AED 842/sqft
  Price gap ratio: 3.4x
